# Stage 4A — Baseline Detector Inference

This notebook:
1. Loads `face_manifest.csv`
2. Runs a baseline deepfake detector on face crops
3. Aggregates frame-level scores to video-level scores
4. Saves:
   - `frame_detector_scores.csv`
   - `video_detector_scores.csv`

Use this notebook **before** the merge + experiments notebook.

## Assumption

This notebook assumes you already have a trained or pretrained **binary deepfake detector checkpoint**.

Typical label convention:
- `0` = real
- `1` = fake

You only need to adapt:
- `MODEL_NAME`
- `CHECKPOINT_PATH`
- checkpoint loading logic if your format differs

In [1]:
from pathlib import Path
import math

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()

SUBSET_NAME = "pilot"   # "pilot" or "final"

FACE_MANIFEST_PATH = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_face_manifest.csv"
FRAME_SCORES_OUT = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_frame_detector_scores.csv"
VIDEO_SCORES_OUT = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_detector_scores.csv"

# Detector config
MODEL_NAME = "tf_efficientnet_b0"   # change if needed
NUM_CLASSES = 1                     # binary sigmoid head
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "baseline_detector.pth"

IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Aggregation
VIDEO_SCORE_MODE = "mean"  # "mean", "max", "topk_mean"
TOPK = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FACE_MANIFEST_PATH:", FACE_MANIFEST_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE:", DEVICE)

PROJECT_ROOT: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness
FACE_MANIFEST_PATH: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/pilot_face_manifest.csv
CHECKPOINT_PATH: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/checkpoints/baseline_detector.pth
DEVICE: cuda


## Load face manifest

In [4]:
assert FACE_MANIFEST_PATH.exists(), f"Face manifest not found: {FACE_MANIFEST_PATH}"
face_df = pd.read_csv(FACE_MANIFEST_PATH)
print(f"Loaded {len(face_df):,} face rows from {FACE_MANIFEST_PATH}")
display(face_df.head())

Loaded 11,926 face rows from /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/pilot_face_manifest.csv


,video_id,frame_id,timestamp_sec,label,split,manipulation_family,manipulation_type,identity,source_subset,video_path,frame_path,face_id,face_path,bbox_x1,bbox_y1,bbox_x2,bbox_y2,det_score,face_width,face_height
0,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000000,0.0,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000000_face0,/home/n.salikhova@innopolis.university/deepfak...,314,50,479,215,1.0,165,165
1,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000001,0.2,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000001_face0,/home/n.salikhova@innopolis.university/deepfak...,317,55,479,217,1.0,162,162
2,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000002,0.4,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000002_face0,/home/n.salikhova@innopolis.university/deepfak...,336,78,495,237,1.0,159,159
3,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000003,0.6,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000003_face0,/home/n.salikhova@innopolis.university/deepfak...,344,85,506,247,1.0,162,162
4,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000004,0.8,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000004_face0,/home/n.salikhova@innopolis.university/deepfak...,353,89,509,245,1.0,156,156


## Dataset

In [5]:
class FaceDetectorDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224):
        self.df = df.reset_index(drop=True).copy()
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = Path(row["face_path"])
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))
        img = img.astype(np.float32) / 255.0

        # Simple ImageNet-style normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std

        img = np.transpose(img, (2, 0, 1))
        img = torch.tensor(img, dtype=torch.float32)

        return {
            "image": img,
            "video_id": row["video_id"],
            "frame_id": row["frame_id"],
            "face_id": row["face_id"],
            "face_path": row["face_path"],
            "label": row.get("label", None),
            "split": row.get("split", None),
            "timestamp_sec": row.get("timestamp_sec", np.nan),
            "manipulation_family": row.get("manipulation_family", None),
            "manipulation_type": row.get("manipulation_type", None),
        }

dataset = FaceDetectorDataset(face_df, image_size=IMAGE_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Dataset size:", len(dataset))

Dataset size: 11926


## Build model

This version uses `timm`.  
If your checkpoint was trained with another architecture, adapt this cell.

In [7]:
!pip install timm -q

In [8]:
import timm

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

# Common checkpoint formats
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

# Remove 'module.' prefix if present
clean_state = {}
for k, v in state.items():
    nk = k.replace("module.", "")
    clean_state[nk] = v

missing, unexpected = model.load_state_dict(clean_state, strict=False)
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model = model.to(DEVICE)
model.eval()
print("Model loaded")

FileNotFoundError: [Errno 2] No such file or directory: '/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/checkpoints/baseline_detector.pth'

## Inference

In [ ]:
@torch.no_grad()
def run_detector_inference(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    rows = []

    for batch in tqdm(loader, total=len(loader), desc="Running detector"):
        images = batch["image"].to(DEVICE)
        logits = model(images)

        if logits.ndim == 2 and logits.shape[1] == 1:
            logits = logits[:, 0]

        probs = torch.sigmoid(logits).detach().cpu().numpy()

        for i in range(len(probs)):
            rows.append({
                "video_id": batch["video_id"][i],
                "frame_id": batch["frame_id"][i],
                "face_id": batch["face_id"][i],
                "face_path": batch["face_path"][i],
                "label": batch["label"][i],
                "split": batch["split"][i],
                "timestamp_sec": float(batch["timestamp_sec"][i]) if str(batch["timestamp_sec"][i]) != "nan" else np.nan,
                "manipulation_family": batch["manipulation_family"][i],
                "manipulation_type": batch["manipulation_type"][i],
                "detector_score": float(probs[i]),
                "detector_pred": int(probs[i] >= 0.5),
            })

    return pd.DataFrame(rows)

frame_scores_df = run_detector_inference(model, loader)
print(f"Generated {len(frame_scores_df):,} frame-level detector scores")
display(frame_scores_df.head())

frame_scores_df.to_csv(FRAME_SCORES_OUT, index=False)
print("Saved:", FRAME_SCORES_OUT)

## Aggregate to video-level scores

In [ ]:
def aggregate_video_scores(df: pd.DataFrame, mode: str = "mean", topk: int = 10) -> pd.DataFrame:
    rows = []

    for video_id, grp in tqdm(df.groupby("video_id"), desc="Aggregating video scores"):
        scores = grp["detector_score"].to_numpy()

        if mode == "mean":
            video_score = float(np.mean(scores))
        elif mode == "max":
            video_score = float(np.max(scores))
        elif mode == "topk_mean":
            k = min(topk, len(scores))
            top_scores = np.sort(scores)[-k:]
            video_score = float(np.mean(top_scores))
        else:
            raise ValueError(f"Unknown aggregation mode: {mode}")

        rows.append({
            "video_id": video_id,
            "label": grp["label"].iloc[0],
            "split": grp["split"].iloc[0],
            "manipulation_family": grp["manipulation_family"].iloc[0],
            "manipulation_type": grp["manipulation_type"].iloc[0],
            "n_face_frames": len(grp),
            "video_score_mode": mode,
            "detector_score": video_score,
            "detector_pred": int(video_score >= 0.5),
        })

    return pd.DataFrame(rows)

video_scores_df = aggregate_video_scores(frame_scores_df, mode=VIDEO_SCORE_MODE, topk=TOPK)
print(f"Generated {len(video_scores_df):,} video-level detector scores")
display(video_scores_df.head())

video_scores_df.to_csv(VIDEO_SCORES_OUT, index=False)
print("Saved:", VIDEO_SCORES_OUT)

## Quick checks

In [ ]:
print("Frame-level rows:", len(frame_scores_df))
print("Video-level rows:", len(video_scores_df))
display(video_scores_df[["label", "split"]].value_counts().rename("count").reset_index())

## Next

After this notebook:
1. Merge
   - `manifest`
   - `video_emotion_features.csv`
   - `detector_scores.csv`
2. Run:
   - baseline metrics
   - emotion analysis
   - fusion model
   - ablation